The conda environment (kernel) for this notebook:
```
conda create -n mapping 
conda activate mapping
conda install ipykernel salmon zstandard seqkit multiqc 
python -m ipykernel install --user --name=mapping
``` 

### Dependencies

In [ ]:
from pathlib import Path
import subprocess
import shutil
import requests
import tqdm
import sys

# --- Import Python Utilities ---
# -------------------------------
relative_target_path = Path("utils") / "python_utils"
project_root = Path.cwd()
found_root = None
while True:
    if (project_root / relative_target_path).is_dir():
        found_root = project_root
        break # Found it!
    # Stop if we reach the filesystem root
    if project_root == project_root.parent:
        raise FileNotFoundError(
            f"Could not find the directory structure '{relative_target_path}'"
        )
    # Go one level up for the next iteration
    project_root = project_root.parent
# Add the found project root to sys.path if it's not already there
if found_root:
    path_str = str(found_root)
    if path_str not in sys.path:
        sys.path.append(path_str)
        print(f"Added '{path_str}' to sys.path")

from utils.python_utils import (
    # Directory Paths
    datasets_dir,
    log_dir,
    # Functions
    execute_command,
    barcode_info_from_filename,
    salmon_genome_index_for_species,
    salmon_map_reads
)

Found structure 'utils/python_utils' within: /home/arnek/workflow_dev
Added '/home/arnek/workflow_dev' to sys.path


### Download input files
 - Acomoys cahirinus reference [GCA_029890205.1](https://ftp.ensembl.org/pub/rapid-release/species/Acomys_cahirinus/GCA_029890205.1/ensembl/)
 - Mus musculus reference genome: [GRCm39](https://www.ensembl.org/Mus_musculus/Info/Index)

The Salmon genome index generation steps follow this official tutorial: https://combine-lab.github.io/alevin-tutorial/2019/selective-alignment/. In this context, cDNA = transcriptome.

In [ ]:
acomys_genome_url = "https://ftp.ensembl.org/pub/rapid-release/species/Acomys_cahirinus/GCA_029890205.1/ensembl/genome/Acomys_cahirinus-GCA_029890205.1-unmasked.fa.gz"
acomys_cdna_url = "https://ftp.ensembl.org/pub/rapid-release/species/Acomys_cahirinus/GCA_029890205.1/ensembl/geneset/2023_11/Acomys_cahirinus-GCA_029890205.1-2023_11-cdna.fa.gz"
mus_genome_url = "https://ftp.ensembl.org/pub/release-113/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz"
mus_cdna_url = "https://ftp.ensembl.org/pub/release-113/fasta/mus_musculus/cdna/Mus_musculus.GRCm39.cdna.all.fa.gz"

reference_dir = Path(datasets_dir) / 'genomic_reference'
acomys_reference_dir = reference_dir / 'Acomys_cahirinus_reference'
mus_reference_dir = reference_dir / 'Mus_musculus_reference'

if acomys_reference_dir.is_dir():
    shutil.rmtree(acomys_reference_dir)
acomys_reference_dir.mkdir(parents=True, exist_ok=False)
if mus_reference_dir.is_dir():
    shutil.rmtree(mus_reference_dir)
mus_reference_dir.mkdir(parents=True, exist_ok=False)

def download_web_file(
        url: str,
        output_file: Path):
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
            # Get file size from headers (in bytes)
            total_size = int(r.headers.get('content-length', 0))
            with open(output_file, 'wb') as f:
                with tqdm.tqdm(
                    total=total_size,
                    unit='B',
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=f"Downloading {output_file.name}",
                    bar_format="{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))
        if not output_file.is_file() or not output_file.stat().st_size > 0:
            print(f"Error: Failed to download {url} to {output_file}")
            return False
        print(f"Successfully downloaded {output_file}")
        return True
    except requests.exceptions.RequestException as e:
        print(f"Error downloading file: {e}")
        return False

# --- Download Mus musculus genome and cDNA ---
# ---------------------------------------------
download_success = download_web_file(
    url=mus_genome_url,
    output_file= Path(mus_reference_dir / 'Mus_musculus_GRCm39_primary_assembly.fasta.gz')
)
if not download_success:
    print("Failed to download Mus musculus genome")
download_success = download_web_file(
    url=mus_cdna_url,
    output_file=Path(mus_reference_dir / 'Mus_musculus_GRCm39_cdna.fasta.gz')
)
if not download_success:
    print("Failed to download Mus musculus cDNA")

# --- Download Acomys genome and cDNA ---
# ---------------------------------------
download_success = download_web_file(
    url=acomys_genome_url,
    output_file=Path(acomys_reference_dir / 'Acomys_cahirinus_GCA_029890205.1_primary_assembly.fasta.gz')
)
if not download_success:
    print("Failed to download Acomys genome")
download_success = download_web_file(
    url=acomys_cdna_url,
    output_file=Path(acomys_reference_dir / 'Acomys_cahirinus_GCA_029890205.1_cdna.fasta.gz')
)
if not download_success:
    print("Failed to download Acomys cDNA")

### Genome index generation (Salmon)
Define functions to generate genome index for a given species.


Generate genome index

In [ ]:
# --- Acomys cahirinus ---
# ------------------------
reference_dir = Path(datasets_dir) / 'genomic_reference'
# Output where species index directories will be generated
genomic_index_dir = reference_dir / 'genomic_reference_index'
genomic_index_dir.mkdir(parents=True, exist_ok=True)

# species input directory with the downloaded cDNA and genome FASTA files
acomys_reference_dir = reference_dir / 'Acomys_cahirinus_reference'
# Input directory cannot be empty
if not acomys_reference_dir.is_dir():
    raise ValueError(f"Acomys cahirinus reference directory not found: {acomys_reference_dir}")
elif next(acomys_reference_dir.iterdir(), None) is None:
    raise ValueError(f"Acomys cahirinus reference directory is empty: {acomys_reference_dir}")

# Generate the genome index
salmon_genome_index_for_species(
    species_name='Acomys cahirinus',
    input_genome_dir=acomys_reference_dir,
    output_genome_index_dir=genomic_index_dir
)

In [ ]:
# --- Mus musculus ---
# --------------------
reference_dir = Path(datasets_dir) / 'genomic_reference'
# Output where species index directories will be generated
genomic_index_dir = reference_dir / 'genomic_reference_index'
genomic_index_dir.mkdir(parents=True, exist_ok=True)

# species input directory with the downloaded cDNA and genome FASTA files
mus_reference_dir = reference_dir / 'Mus_musculus_reference'
# Input directory cannot be empty
if not mus_reference_dir.is_dir():
    raise ValueError(f"Mus musculus reference directory not found: {mus_reference_dir}")
elif next(mus_reference_dir.iterdir(), None) is None:
    raise ValueError(f"Mus musculus reference directory is empty: {mus_reference_dir}")

# Generate the genome index
salmon_genome_index_for_species(
    species_name='Mus musculus',
    input_genome_dir=mus_reference_dir,
    output_genome_index_dir=genomic_index_dir
)

### Mapping

In [ ]:
# --- Mapping Acomys cahirinus ---
# --------------------------------
# Fetch the reference genome index
reference_dir = Path(datasets_dir) / 'genomic_reference'
genomic_index_dir = reference_dir / 'genomic_reference_index'
acomys_index_dir = genomic_index_dir / 'Acomys_cahirinus_genome_index'
if not acomys_index_dir.is_dir():
    raise FileNotFoundError(f"Genomic index directory not found: {acomys_index_dir}")

# Fetch the R2 demultiplexed reads
demultiplexed_reads_dir = Path(datasets_dir) / 'Tomoseq_reads_qc' / 'further_processed' / 'demultiplexed_reads'
if not demultiplexed_reads_dir.is_dir():
    raise FileNotFoundError(f"Demultiplexed reads directory not found: {demultiplexed_reads_dir}")
demultiplexed_r2_dir_list = demultiplexed_reads_dir.rglob('*R2_AC-*demultiplexed*')
demultiplexed_r2_dir_list = [path for path in demultiplexed_r2_dir_list if path.is_dir()]
if not demultiplexed_r2_dir_list:
    raise FileNotFoundError(f"No R2 demultiplex directories found in {demultiplexed_reads_dir}")
print(f"Found {len(demultiplexed_r2_dir_list)} R2 demultiplex directories")

mapped_dir = Path(datasets_dir) / 'Tomoseq_mapped'

# Iterate over the R2 demultiplex directories
for dir_path in demultiplexed_r2_dir_list:
    sample_id = dir_path.name.split('_')[1]
    print(f"Processing sample {sample_id}")

    # Format output
    sample_mapped_dir = mapped_dir / f"{sample_id}_mapped"
    sample_mapped_dir.mkdir(parents=True, exist_ok=True)

    # Fetch the demultiplexed input files
    input_fastq_file_list = dir_path.rglob('*bcode_*.dmplx.*')
    input_fastq_file_list = [path for path in input_fastq_file_list if path.is_file()]
    if not input_fastq_file_list:
        raise FileNotFoundError(f"No input FASTQ files found in {dir_path}")
    
    # Iterate over the input FASTQ files
    for input_fastq_file in input_fastq_file_list:
        print(f"Processing file {input_fastq_file.name}")
        barcode, barcode_id = barcode_info_from_filename(input_fastq_file.name)
        print(f"Barcode: {barcode}, Barcode ID: {barcode_id}")
        
        output_dir_path = sample_mapped_dir / f"bcode_{barcode_id}_{barcode}"
        # Skip if the output directory already exists
        if output_dir_path.is_dir():
            print(f"Skipping {output_dir_path} because it already exists")
            continue

        salmon_map_reads(
            input_reads_fastq=input_fastq_file,
            input_index_dir=acomys_index_dir,
            output_path_prefix=sample_mapped_dir / f"bcode_{barcode_id}_{barcode}",
            mapping_threads=4
        )


#### Mapping result summary per sample ID. 

In [ ]:
mapped_dir = Path(datasets_dir) / 'Tomoseq_mapped'
if not mapped_dir.is_dir():
    raise FileNotFoundError(f'Mapped data not found in {mapped_dir}')

sample_mapped_dir_list = mapped_dir.glob('*/')
sample_mapped_dir_list = [x for x in sample_mapped_dir_list if x.is_dir()]
if not sample_mapped_dir_list:
    raise FileNotFoundError(f'No mapped data found in {mapped_dir}')

multiqc_salmon_dir = log_dir / 'multiqc_report' / 'multiqc_salmon_mapping'
multiqc_salmon_dir.mkdir(exist_ok=True, parents=True)

# --- Iterate over the sample directories ---
# -------------------------------------------
for sample_dir in sample_mapped_dir_list:
    sample_id = sample_dir.name.split('_')[0]
    print(f'Processing sample {sample_id}')

    # Fetch the barcode directories
    barcode_dir_list = sample_dir.glob('bcode_*')
    barcode_dir_list = [x for x in barcode_dir_list if x.is_dir()]
    if not barcode_dir_list:
        print(f'No barcode directories found for sample {sample_id}')
        continue
    else:
        print(f'Found {len(barcode_dir_list)} barcode directories for sample {sample_id}')
                    # Run MultiQC
        multiqc_cmd = ['multiqc', 
                        '--outdir', multiqc_salmon_dir,
                        '--filename', f'{sample_id}_MultiQC_Salmon_mapping.html', 
                        '--title', f'Salmon mapping for {sample_id}', 
                        sample_dir]
        exit_status = execute_command(multiqc_cmd)
        if exit_status != 0:
            raise subprocess.CalledProcessError(exit_status, multiqc_cmd)
